Inspired by: https://github.com/timqqt/FinRL-Library/blob/master/FinRL_portfolio_allocation_NeurIPS_2020.ipynb

<a id='1.1'></a>
## 1. Install all the packages through FinRL library


In [1]:
## install finrl library
!pip install wrds
!pip install swig
!pip install -q condacolab
import condacolab
condacolab.install()
!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 25.6 MB/s eta 0:00:00
⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libgl1-mesa-glx:amd64.
(Reading database ... 124947 files and directories currently installed.)
Preparing to unpack .../libgl1-mesa-glx_23.0.4-0ubuntu1~22.04.1_amd64.deb ...
Unpacking libgl1-mesa-glx:amd64 (23.0.4-0ubuntu1~22.04.1) ...
Selecting previously unselected package swig4.0.
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.


<a id='1.2'></a>
## 1.2. Check if the additional packages needed are present, if not install them.
* Yahoo Finance API
* pandas
* numpy
* matplotlib
* stockstats
* OpenAI gym
* stable-baselines
* tensorflow
* pyfolio

<a id='1.3'></a>
## 1.3. Import Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
%matplotlib inline
import datetime

from finrl import config
from finrl import config_tickers
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_portfolio_allocation.env_portfolio import StockPortfolioEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline,convert_daily_return_to_pyfolio_ts
from finrl.meta.data_processor import DataProcessor
from finrl.meta.data_processors.processor_yahoofinance import YahooFinanceProcessor
import sys
sys.path.append("../FinRL-Library")

/usr/local/lib/python3.11/site-packages/pandas_datareader/compat/__init__.py:11: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  PANDAS_VERSION = LooseVersion(pd.__version__)
/usr/local/lib/python3.11/site-packages/pyfolio/pos.py:26: UserWarning: Module "zipline.assets" not found; mutltipliers will not be applied to position notionals.
  warnings.warn(


<a id='1.4'></a>
## 1.4. Create Folders

In [2]:
import os
if not os.path.exists("./" + config.DATA_SAVE_DIR):
    os.makedirs("./" + config.DATA_SAVE_DIR)
if not os.path.exists("./" + config.TRAINED_MODEL_DIR):
    os.makedirs("./" + config.TRAINED_MODEL_DIR)
if not os.path.exists("./" + config.TENSORBOARD_LOG_DIR):
    os.makedirs("./" + config.TENSORBOARD_LOG_DIR)
if not os.path.exists("./" + config.RESULTS_DIR):
    os.makedirs("./" + config.RESULTS_DIR)

## 2. Generate synthethic data

In [3]:


# Generate synthetic stock prices again
def generate_stock_prices(days=252, initial_prices={"AAPL": 150, "MSFT": 300}):
    np.random.seed(42)

    # Define trends
    drift = {"AAPL": 0.002, "MSFT": -0.001}  # AAPL trends up, MSFT trends down
    volatility = {"AAPL": 0.4, "MSFT": 0.2}  # Same volatility

    # Store price series
    stock_prices = {stock: [initial_prices[stock]] for stock in initial_prices.keys()}

    for i in range(1, days):
        for stock in initial_prices.keys():
            log_return = np.random.normal(drift[stock], volatility[stock])
            stock_prices[stock].append(stock_prices[stock][-1] * np.exp(log_return))

    return pd.DataFrame({"date": pd.date_range(start="2023-01-01", periods=days),
                         "AAPL": stock_prices["AAPL"], "MSFT": stock_prices["MSFT"]})

# Generate the dataset again
df = generate_stock_prices()

# Reshape the DataFrame to have "tic", "close", and "date" as separate columns
df_melted = df.melt(id_vars=["date"], value_vars=["AAPL", "MSFT"], var_name="tic", value_name="close")

# Compute past returns AFTER melting
df_melted.sort_values(by=["tic", "date"], inplace=True)

# Calculate past returns for each stock separately
df_melted["return_t-1"] = df_melted.groupby("tic")["close"].pct_change(1)
df_melted["return_t-2"] = df_melted.groupby("tic")["close"].pct_change(2)
df_melted["return_t-3"] = df_melted.groupby("tic")["close"].pct_change(3)
df_melted["return_t-4"] = df_melted.groupby("tic")["close"].pct_change(4)
df_melted["return_t-5"] = df_melted.groupby("tic")["close"].pct_change(5)

# Drop NaN values caused by pct_change()
df_melted = df_melted.dropna().reset_index(drop=True)


In [4]:
# Plot price trends for AAPL and MSFT
plt.figure(figsize=(12, 6))
for stock in df_melted["tic"].unique():
    stock_data = df_melted[df_melted["tic"] == stock]
    plt.plot(stock_data["date"], stock_data["close"], label=stock)

# Formatting the plot
plt.xlabel("Date")
plt.ylabel("Stock Price (Close)")
plt.title("Stock Price Trends: AAPL vs MSFT")
plt.legend()
plt.grid(True)

# Save the plot as an image
plt.savefig("stock_price_trends.png")

# Display the plot
plt.show()

In [5]:
df = df_melted

## Training data split

In [6]:
df['date'].min()

Timestamp('2023-01-06 00:00:00')

In [7]:
df['date'].max()

Timestamp('2023-09-09 00:00:00')

In [8]:
train = data_split(df, '2023-01-06','2023-09-09')
#trade = data_split(df, '2020-01-01', config.END_DATE)

In [9]:
train.head()

,date,tic,close,return_t-1,return_t-2,return_t-3,return_t-4,return_t-5
0,2023-01-06,AAPL,339.892840,-0.169552,0.565015,0.427942,0.853933,1.265952
0,2023-01-06,MSFT,488.288856,0.113504,0.296929,0.236360,0.674940,0.627630
1,2023-01-07,AAPL,282.948051,-0.167537,-0.308683,0.302816,0.188708,0.543330
1,2023-01-07,MSFT,444.416061,-0.089850,0.013456,0.180400,0.125273,0.524447
2,2023-01-08,AAPL,312.326222,0.103829,-0.081104,-0.236904,0.438086,0.312130


## Environment for Portfolio Allocation


In [19]:
import numpy as np
import pandas as pd
from gym.utils import seeding
import gym
from gym import spaces
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from stable_baselines3.common.vec_env import DummyVecEnv
class StockPortfolioEnv(gym.Env):
    """A single stock trading environment for OpenAI gym

    Attributes
    ----------
        df: DataFrame
            input data
        stock_dim : int
            number of unique stocks
        hmax : int
            maximum number of shares to trade
        initial_amount : int
            start money
        transaction_cost_pct: float
            transaction cost percentage per trade
        reward_scaling: float
            scaling factor for reward, good for training
        state_space: int
            the dimension of input features
        action_space: int
            equals stock dimension
        tech_indicator_list: list
            a list of technical indicator names
        turbulence_threshold: int
            a threshold to control risk aversion
        day: int
            an increment number to control date

    Methods
    -------
    _sell_stock()
        perform sell action based on the sign of the action
    _buy_stock()
        perform buy action based on the sign of the action
    step()
        at each step the agent will return actions, then
        we will calculate the reward, and return the next observation.
    reset()
        reset the environment
    render()
        use render to return other functions
    save_asset_memory()
        return account value at each time step
    save_action_memory()
        return actions/positions at each time step


    """
    def __init__(self,
                df,
                stock_dim,
                hmax,
                initial_amount,
                transaction_cost_pct,
                reward_scaling,
                state_space,
                action_space,
                tech_indicator_list,  # now e.g., ["return_t-1", "return_t-2", "return_t-3", "return_t-4", "return_t-5"]
                turbulence_threshold=None,
                lookback=252,
                day=0):
        self.day = day
        self.lookback = lookback
        self.df = df
        self.stock_dim = stock_dim
        self.hmax = hmax
        self.initial_amount = initial_amount
        self.transaction_cost_pct = transaction_cost_pct
        self.reward_scaling = reward_scaling
        self.state_space = state_space
        self.action_space = action_space
        self.tech_indicator_list = tech_indicator_list

        self.action_space = spaces.Box(low=0, high=1, shape=(self.action_space,))
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.state_space + len(self.tech_indicator_list), self.state_space))

        # NEW: Load data for the current day based on "date"
        self.data = self.df[self.df["date"] == self.df["date"].unique()[self.day]]
        self.state = self._get_state()  # Get state from our new helper
        self.terminal = False
        self.turbulence_threshold = turbulence_threshold
        self.portfolio_value = self.initial_amount
        self.asset_memory = [self.initial_amount]
        self.portfolio_return_memory = [0]
        self.actions_memory = [[1 / self.stock_dim] * self.stock_dim]
        self.date_memory = [self.data.date.unique()[0]]

    def _get_state(self):
        # Sort by ticker for consistent ordering.
        data_sorted = self.data.sort_values("tic")
        state_columns = ["close"] + self.tech_indicator_list
        state_array = data_sorted[state_columns].to_numpy()
        return state_array.flatten()

    def normalize_actions(self, actions):
        """
        Normalize action values to ensure the portfolio weights sum to 1.
        - Makes sure all weights are non-negative.
        - If all actions are negative, sets default allocation to first stock.
        """
        actions = np.maximum(actions, 0)  # Ensure no negative allocations
        total = actions.sum()

        if total == 0:
            actions[0] = 1  # If all are 0, allocate everything to the first stock

        return actions / actions.sum()  # Normalize to sum to 1

    def step(self, actions):
        self.terminal = self.day >= len(self.df["date"].unique()) - 1
        print(self.day)
        if self.terminal:
                    df = pd.DataFrame(self.portfolio_return_memory)
                    df.columns = ['daily_return']
                    plt.plot(df.daily_return.cumsum(),'r')
                    plt.savefig('results/cumulative_reward.png')
                    plt.close()

                    plt.plot(self.portfolio_return_memory,'r')
                    plt.savefig('results/rewards.png')
                    plt.close()

                    print("=================================")
                    print("begin_total_asset:{}".format(self.asset_memory[0]))
                    print("end_total_asset:{}".format(self.portfolio_value))

                    df_daily_return = pd.DataFrame(self.portfolio_return_memory)
                    df_daily_return.columns = ['daily_return']
                    if df_daily_return['daily_return'].std() !=0:
                      sharpe = (252**0.5)*df_daily_return['daily_return'].mean()/ \
                              df_daily_return['daily_return'].std()
                      print("Sharpe: ",sharpe)
                    print("=================================")

                    return self.state, self.reward, self.terminal,{}

        else:
            print("Raw Actions:", actions)

            weights = self.normalize_actions(actions)

            #weights = self.softmax_normalization(actions)  # Ensure allocations sum to 1
            self.actions_memory.append(weights)
            last_day_memory = self.data
            print(weights)
            self.day += 1
            self.data = self.df[self.df["date"] == self.df["date"].unique()[self.day]]
            self.state = self._get_state()

            # Compute individual stock returns
            stock_returns = (self.data.close.values / last_day_memory.close.values) - 1

            # Compute portfolio return (weighted sum of stock returns)
            portfolio_return = sum(stock_returns * weights)

            # **NEW REWARD FUNCTION**
            # Penalize holding the losing stock (MSFT) and reward AAPL investments
            bad_stock_penalty = sum((weights * (stock_returns < 0)) * abs(stock_returns))  # Punish negative return stocks
            good_stock_reward = sum((weights * (stock_returns > 0)) * stock_returns)  # Reward positive return stocks

            # Assign the new reward function (Normalized for better learning)
            self.reward = self.reward = (good_stock_reward - bad_stock_penalty) * 500 # <-- ADD THIS LINE HERE


            # Update portfolio value
            new_portfolio_value = self.portfolio_value * (1 + portfolio_return)
            self.portfolio_value = new_portfolio_value
            print(self.portfolio_value)
            # Store historical data
            self.portfolio_return_memory.append(portfolio_return)
            self.date_memory.append(self.data.date.unique()[0])
            self.asset_memory.append(new_portfolio_value)

        return self.state, self.reward, self.terminal, {}


    def reset(self):
        self.asset_memory = [self.initial_amount]
        self.day = 0
        self.data = self.df[self.df["date"] == self.df["date"].unique()[self.day]]
        self.state = self._get_state()
        self.portfolio_value = self.initial_amount
        self.terminal = False
        self.portfolio_return_memory = [0]
        self.actions_memory = [[1 / self.stock_dim] * self.stock_dim]
        self.date_memory = [self.data.date.unique()[0]]
        return self.state

    def softmax_normalization(self, actions):
        numerator = np.exp(actions)
        denominator = np.sum(np.exp(actions))
        softmax_output = numerator/denominator
        return softmax_output


    def save_asset_memory(self):
        date_list = self.date_memory
        portfolio_return = self.portfolio_return_memory
        #print(len(date_list))
        #print(len(asset_list))
        df_account_value = pd.DataFrame({'date':date_list,'daily_return':portfolio_return})
        return df_account_value

    def save_action_memory(self):
        date_list = self.date_memory
        df_date = pd.DataFrame(date_list, columns=['date'])
        action_list = self.actions_memory
        df_actions = pd.DataFrame(action_list)
        # NEW: Set columns based on sorted unique tickers
        df_actions.columns = self.data.sort_values("tic")["tic"].unique()
        df_actions.index = df_date.date
        return df_actions




    def _seed(self, seed=None):
        self.np_random, seed = seeding.np_random(seed)
        return [seed]

    def get_sb_env(self):
        e = DummyVecEnv([lambda: self])
        obs = e.reset()
        return e, obs

In [20]:
stock_dimension = len(train.tic.unique())
state_space = stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")


Stock Dimension: 2, State Space: 2


In [21]:
stock_dimension = len(train["tic"].unique())  # Count unique stocks
state_space = len(["close", "return_t-1", "return_t-2", "return_t-3", "return_t-4", "return_t-5"]) * stock_dimension

env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "transaction_cost_pct": 0.001,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": ["return_t-1", "return_t-2", "return_t-3", "return_t-4", "return_t-5"],
    "action_space": stock_dimension,
    "reward_scaling": 1e-4
}

# Create the environment
e_train_gym = StockPortfolioEnv(df=train, **env_kwargs)


In [22]:
pip install shimmy>=2.0

In [23]:
env_train, _ = e_train_gym.get_sb_env()
print(type(env_train))

<class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>


/usr/local/lib/python3.11/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


<a id='5'></a>
#Implement DRL Algorithms
* The implementation of the DRL algorithms are based on **OpenAI Baselines** and **Stable Baselines**. Stable Baselines is a fork of OpenAI Baselines, with a major structural refactoring, and code cleanups.
* FinRL library includes fine-tuned standard DRL algorithms, such as DQN, DDPG,
Multi-Agent DDPG, PPO, SAC, A2C and TD3. We also allow users to
design their own DRL algorithms by adapting these DRL algorithms.

In [24]:
# initialize
agent = DRLAgent(env = env_train)

### Model 1: **A2C**


In [ ]:
agent = DRLAgent(env = env_train)

A2C_PARAMS = {"n_steps": 5, "ent_coef": 0.005, "learning_rate": 0.0002}
model_a2c = agent.get_model(model_name="a2c",model_kwargs = A2C_PARAMS)

{'n_steps': 5, 'ent_coef': 0.005, 'learning_rate': 0.0002}
Using cpu device


In [ ]:
trained_a2c = agent.train_model(model=model_a2c,
                                tb_log_name='a2c',
                                total_timesteps=50000)

AttributeError: 'StockPortfolioEnv' object has no attribute 'softmax_normalization'

In [ ]:
trained_a2c.save('/content/trained_models/trained_a2c.zip')

### Model 2: **PPO**


In [ ]:
agent = DRLAgent(env = env_train)
PPO_PARAMS = {
    "n_steps": 2048,
    "ent_coef": 0.005,
    "learning_rate": 0.0001,
    "batch_size": 128,
}
model_ppo = agent.get_model("ppo",model_kwargs = PPO_PARAMS)

In [ ]:
trained_ppo = agent.train_model(model=model_ppo,
                             tb_log_name='ppo',
                             total_timesteps=80000)

In [ ]:
trained_ppo.save('/content/trained_models/trained_ppo.zip')

### Model 3: **DDPG**


In [27]:
train

,date,tic,close,return_t-1,return_t-2,return_t-3,return_t-4,return_t-5
0,2023-01-06,AAPL,339.892840,-0.169552,0.565015,0.427942,0.853933,1.265952
0,2023-01-06,MSFT,488.288856,0.113504,0.296929,0.236360,0.674940,0.627630
1,2023-01-07,AAPL,282.948051,-0.167537,-0.308683,0.302816,0.188708,0.543330
1,2023-01-07,MSFT,444.416061,-0.089850,0.013456,0.180400,0.125273,0.524447
2,2023-01-08,AAPL,312.326222,0.103829,-0.081104,-0.236904,0.438086,0.312130
...,...,...,...,...,...,...,...,...
243,2023-09-06,MSFT,557.997660,0.112692,-0.064314,0.256731,0.497567,0.205064
244,2023-09-07,AAPL,454.224826,-0.338271,0.219391,0.260235,0.258738,1.476797
244,2023-09-07,MSFT,536.618296,-0.038314,0.070060,-0.100164,0.208580,0.440189
245,2023-09-08,AAPL,320.648333,-0.294076,-0.532869,-0.139203,-0.110369,-0.111426


In [25]:
agent = DRLAgent(env = env_train)
DDPG_PARAMS = {
    "batch_size": 128,
    "buffer_size": 100000,  # Increase buffer size for more diverse experience replay
    "learning_rate": 0.0005,  # Reduce learning rate for more stable updates
    "tau": 0.02,  # Increase tau for slower target updates
    "gamma": 0.99,  # Discount factor
    "train_freq": (1, "episode")  # Train every step
}
DDPG_PARAMS = {"batch_size": 128, "buffer_size": 50000, "learning_rate": 0.001}


model_ddpg = agent.get_model("ddpg",model_kwargs = DDPG_PARAMS)

{'batch_size': 128, 'buffer_size': 50000, 'learning_rate': 0.001}
Using cpu device


In [28]:
trained_ddpg = agent.train_model(model=model_ddpg,
                             tb_log_name='ddpg',
                             total_timesteps=345)

0
Raw Actions: [0.26923457 0.897165  ]
[0.23082532 0.76917464]
892217.7020032405
1
Raw Actions: [0.22796321 0.0903092 ]
[0.71625185 0.28374812]
877902.7243689494
2
Raw Actions: [0.56064093 0.2950348 ]
[0.65520257 0.34479743]
559327.5153666124
3
Raw Actions: [0.5720756 0.5932441]
[0.49091733 0.5090827 ]
486394.4534599485
4
Raw Actions: [0.07746869 0.24199533]
[0.24249582 0.75750417]
359693.8211569891
5
Raw Actions: [0.70571965 0.4187937 ]
[0.6275778  0.37242216]
534432.3261711636
6
Raw Actions: [0.6203729  0.17383066]
[0.78112584 0.2188742 ]
517629.48968998046
7
Raw Actions: [0.23598641 0.0509133 ]
[0.82253975 0.17746028]
436968.7036069963
8
Raw Actions: [0.61200523 0.49022153]
[0.5552444 0.4447556]
362710.9467823863
9
Raw Actions: [0.17523617 0.29220015]
[0.3748878  0.62511224]
320821.46076526446
10
Raw Actions: [0.5207224 0.3828428]
[0.5762975  0.42370248]
342317.84518544417
11
Raw Actions: [0.9534229 0.8997251]
[0.5144883  0.48551172]
309897.4386085317
12
Raw Actions: [0.01008874 0.9

In [ ]:
trained_ddpg.save('/content/trained_models/trained_ddpg.zip')

### Model 4: **SAC**


In [ ]:
agent = DRLAgent(env = env_train)
SAC_PARAMS = {
    "batch_size": 128,
    "buffer_size": 100000,
    "learning_rate": 0.0003,
    "learning_starts": 100,
    "ent_coef": "auto_0.1",
}

model_sac = agent.get_model("sac",model_kwargs = SAC_PARAMS)

In [ ]:
trained_sac = agent.train_model(model=model_sac,
                             tb_log_name='sac',
                             total_timesteps=50000)

In [ ]:
trained_sac.save('/content/trained_models/trained_sac.zip')

### Model 5: **TD3**


In [ ]:
agent = DRLAgent(env = env_train)
TD3_PARAMS = {"batch_size": 100,
              "buffer_size": 1000000,
              "learning_rate": 0.001}

model_td3 = agent.get_model("td3",model_kwargs = TD3_PARAMS)

In [ ]:
trained_td3 = agent.train_model(model=model_td3,
                             tb_log_name='td3',
                             total_timesteps=30000)

In [ ]:
trained_td3.save('/content/trained_models/trained_td3.zip')

In [ ]:
trade = data_split(df,'2020-07-01', '2021-10-31')
e_trade_gym = StockPortfolioEnv(df = trade, **env_kwargs)


In [ ]:
trade.shape

(9436, 19)

In [ ]:
df_daily_return, df_actions = DRLAgent.DRL_prediction(model=trained_a2c,
                        environment = e_trade_gym)

begin_total_asset:1000000
end_total_asset:1414639.0988650718
Sharpe:  1.928505358206197
hit end!


In [ ]:
df_daily_return.head()

,date,daily_return
0,2020-07-01,0.000000
1,2020-07-02,0.004836
2,2020-07-06,0.016306
3,2020-07-07,-0.015250
4,2020-07-08,0.006693


In [ ]:
df_daily_return.to_csv('df_daily_return.csv')

In [ ]:
df_actions.head()

,AAPL,AMGN,AXP,BA,CAT,CRM,CSCO,CVX,DIS,GS,...,MMM,MRK,MSFT,NKE,PG,TRV,UNH,VZ,WBA,WMT
date,,,,,,,,,,,,,,,,,,,,,
2020-07-01,0.035714,0.035714,0.035714,0.035714,0.035714,0.035714,0.035714,0.035714,0.035714,0.035714,...,0.035714,0.035714,0.035714,0.035714,0.035714,0.035714,0.035714,0.035714,0.035714,0.035714
2020-07-02,0.039087,0.021217,0.036143,0.021217,0.041956,0.044596,0.021217,0.056761,0.021217,0.043837,...,0.029670,0.021217,0.057673,0.037758,0.021217,0.021217,0.021413,0.057014,0.057673,0.021217
2020-07-06,0.039087,0.021217,0.036143,0.021217,0.041956,0.044596,0.021217,0.056761,0.021217,0.043837,...,0.029670,0.021217,0.057673,0.037758,0.021217,0.021217,0.021413,0.057014,0.057673,0.021217
2020-07-07,0.039093,0.021209,0.036193,0.021209,0.041960,0.044606,0.021209,0.056802,0.021209,0.043838,...,0.029670,0.021209,0.057653,0.037832,0.021209,0.021209,0.021402,0.057032,0.057653,0.021209
2020-07-08,0.039087,0.021217,0.036143,0.021217,0.041956,0.044596,0.021217,0.056761,0.021217,0.043837,...,0.029670,0.021217,0.057673,0.037758,0.021217,0.021217,0.021413,0.057014,0.057673,0.021217


In [ ]:
df_actions.to_csv('df_actions.csv')

In [ ]:
from pyfolio import timeseries
DRL_strat = convert_daily_return_to_pyfolio_ts(df_daily_return)
perf_func = timeseries.perf_stats
perf_stats_all = perf_func( returns=DRL_strat,
                              factor_returns=DRL_strat,
                                positions=None, transactions=None, turnover_denom="AGB")

In [ ]:
print("==============DRL Strategy Stats===========")
perf_stats_all

==============DRL Strategy Stats===========


Annual return          0.296131
Cumulative returns     0.414639
Annual volatility      0.139606
Sharpe ratio           1.928505
Calmar ratio           3.347257
Stability              0.917786
Max drawdown          -0.088470
Omega ratio            1.380058
Sortino ratio          2.939907
Skew                  -0.111808
Kurtosis               1.453386
Tail ratio             1.095889
Daily value at risk   -0.016520
Alpha                  0.000000
Beta                   1.000000
dtype: float64

In [ ]:
#baseline stats
print("==============Get Baseline Stats===========")
baseline_df = get_baseline(
        ticker="^DJI",
        start = df_daily_return.loc[0,'date'],
        end = df_daily_return.loc[len(df_daily_return)-1,'date'])

stats = backtest_stats(baseline_df, value_col_name = 'close')

==============Get Baseline Stats===========
[*********************100%***********************]  1 of 1 completed
Shape of DataFrame:  (337, 8)
Annual return          0.275227
Cumulative returns     0.384211
Annual volatility      0.138965
Sharpe ratio           1.824949
Calmar ratio           3.081783
Stability              0.919239
Max drawdown          -0.089308
Omega ratio            1.354658
Sortino ratio          2.705587
Skew                        NaN
Kurtosis                    NaN
Tail ratio             1.053708
Daily value at risk   -0.016502
dtype: float64
